# Week 8: Decision Trees (ID3)

## Student Practice Notebook

**Name:** Aditya Kumar
**Roll Number:** AP24110010333
**Date:** 10-10-2026

## Learning Objectives

After completing this lab, you will be able to:

- Compute entropy and information gain, by hand and in code
- Build an ID3 decision tree by hand on a small table (the exam-style problem)
- Fit and read a `DecisionTreeClassifier`
- Explain how tree depth controls overfitting (same idea as K in Week 7 and degree in Week 6)
- Explain how a tree differs from KNN: it asks questions about *features*, it does not measure *distance*

### Why this week matters
KNN (Week 7) classified by "who is closest?". A decision tree classifies by asking a chain of yes/no questions about features, and you can read the questions. That readability is why trees are used in credit, healthcare and fraud work, and why every ensemble in Unit V is built from them.


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score

%matplotlib inline
np.random.seed(1)


## Part 1 — Entropy and Information Gain

**Entropy** measures how mixed a set of labels is: $H = -\sum p_i \log_2 p_i$. A pure set (all one class) has entropy 0. A 50/50 two-class set has entropy 1.

**Information gain** of splitting on an attribute = entropy before − weighted average entropy after. ID3 always picks the attribute with the highest gain.

### 1.1 Demo: entropy in code


In [ ]:
def entropy(counts):
    counts = np.array(counts, dtype=float)
    p = counts[counts > 0] / counts.sum()
    return float(-np.sum(p * np.log2(p))) + 0.0

print('Pure set      [10, 0]:', entropy([10, 0]))
print('50/50 set     [5, 5] :', entropy([5, 5]))
print('Skewed set    [9, 1] :', entropy([9, 1]))


**Answer:** The 70/30 split has higher entropy than the 90/10 split. The 70/30 split is more mixed, so guessing the label is less certain. Entropy is highest when classes are evenly balanced and lower when one class dominates.


### Student Practice — entropy of a split

A parent node has 8 Yes and 6 No. A candidate split sends 6 samples to the left child (6 Yes, 0 No) and 8 samples to the right child (2 Yes, 6 No). Compute the entropy of the parent and the information gain of the split.


In [ ]:
# Parent: 8 Yes, 6 No. Children: left (6 Yes, 0 No), right (2 Yes, 6 No).
parent_entropy = entropy([8, 6])
left_entropy = entropy([6, 0])
right_entropy = entropy([2, 6])
weighted_child_entropy = (6 / 14) * left_entropy + (8 / 14) * right_entropy
gain = parent_entropy - weighted_child_entropy

print(f"Parent entropy: {parent_entropy:.4f}")
print(f"Left child entropy: {left_entropy:.4f}")
print(f"Right child entropy: {right_entropy:.4f}")
print(f"Weighted child entropy: {weighted_child_entropy:.4f}")
print(f"Information gain: {gain:.4f}")


## Part 2 — ID3 by Hand (the exam-style problem)

Classic 14-day "play tennis" table. Target: `Play`.


In [ ]:
tennis = pd.DataFrame({
    'Outlook':     ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity':    ['High','High','High','High','Normal','Normal','Normal','High','Normal','Normal','Normal','High','Normal','High'],
    'Wind':        ['Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
    'Play':        ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No'],
})
tennis


In [ ]:
def info_gain(df, attribute, target='Play'):
    total = entropy(df[target].value_counts())
    weighted = 0
    for value, subset in df.groupby(attribute):
        weighted += len(subset) / len(df) * entropy(subset[target].value_counts())
    return total - weighted

print('Root entropy:', round(entropy(tennis['Play'].value_counts()), 4))
print('Gain(Wind)  :', round(info_gain(tennis, 'Wind'), 4))


### Student Practice — find the root

1. Compute the information gain for `Outlook`, `Temperature` and `Humidity` using `info_gain`.
2. Which attribute does ID3 pick as the root?
3. For the root's branch that is **not** pure, which attribute would you split on next? (Filter `tennis` to that branch and recompute gains.)


In [ ]:
# Compute information gain for the requested attributes.
for attribute in ["Outlook", "Temperature", "Humidity"]:
    print(f"Gain({attribute}): {info_gain(tennis, attribute):.4f}")

root = max(["Outlook", "Temperature", "Humidity", "Wind"],
           key=lambda attribute: info_gain(tennis, attribute))
print("\nID3 root:", root)

# Outlook's Overcast branch is pure (all Play=Yes); inspect the Sunny and Rain branches.
for branch_value in ["Sunny", "Rain"]:
    subset = tennis[tennis["Outlook"] == branch_value]
    print(f"\nBranch {branch_value}:")
    print(subset[["Outlook", "Play"]].to_string(index=False))
    if subset["Play"].nunique() > 1:
        remaining = [a for a in ["Temperature", "Humidity", "Wind"]]
        for attribute in remaining:
            print(f"Gain({attribute}): {info_gain(subset, attribute):.4f}")


**Answer:** A pure branch contains examples from only one target class. There is no uncertainty left to reduce, so its entropy is 0 and further splitting is unnecessary. ID3 can label that branch directly.


## Part 3 — `DecisionTreeClassifier` on Iris

Same Iris split as Week 7, so you can compare directly. Note: **trees do not need feature scaling**, because they compare a feature to a threshold, not distances between points. That is a real difference from KNN.


In [ ]:
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

tree = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=1)
tree.fit(X_train, y_train)
print('Test accuracy:', accuracy_score(y_test, tree.predict(X_test)))

plt.figure(figsize=(12, 6))
plot_tree(tree, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.show()


### Student Practice — depth sweep

Sweep `max_depth` from 1 to 15. Record train and test accuracy for each, plot both, and report the depth with the best test accuracy.


In [ ]:
depths = list(range(1, 16))
train_accuracies = []
test_accuracies = []

for depth in depths:
    model = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=1)
    model.fit(X_train, y_train)
    train_accuracies.append(accuracy_score(y_train, model.predict(X_train)))
    test_accuracies.append(accuracy_score(y_test, model.predict(X_test)))

results = pd.DataFrame({
    "max_depth": depths,
    "train_accuracy": train_accuracies,
    "test_accuracy": test_accuracies
})
display(results)

plt.figure(figsize=(9, 5))
plt.plot(depths, train_accuracies, marker="o", label="Train accuracy")
plt.plot(depths, test_accuracies, marker="s", label="Test accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.xticks(depths)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Decision tree depth sweep on Iris")
plt.show()

best_index = int(np.argmax(test_accuracies))
print(f"Best test accuracy: {test_accuracies[best_index]:.4f} at max_depth={depths[best_index]}")


**Answer:** The corresponding knob in Week 6 is polynomial degree in polynomial regression, and in Week 7 it is K in KNN. A very small depth can underfit because the tree is too simple. A very large depth can overfit by memorizing training examples; training accuracy may reach 100% while test accuracy stops improving.


# Mini Project A: Digits, Again (Image Track — continuing from Week 7)

Last week KNN reached about 98% on the digits dataset. Now fit a decision tree on the **same split** and ask two new questions: how does it compare, and **which pixels does the tree actually use?**


In [ ]:
digits = load_digits()
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=1, stratify=digits.target)

# KNN baseline from Week 7 (scaled, K=3)
sc = StandardScaler().fit(Xd_train)
knn = KNeighborsClassifier(n_neighbors=3).fit(sc.transform(Xd_train), yd_train)
print('KNN  test accuracy:', accuracy_score(yd_test, knn.predict(sc.transform(Xd_test))))


### Student Practice

1. Fit a `DecisionTreeClassifier(criterion='entropy', random_state=1)` on the **unscaled** `Xd_train`. Print test accuracy. (Why is no scaling needed?)
2. Try `max_depth` values 3, 6, 10 and report test accuracy for each.
3. Take `tree.feature_importances_` (64 values), reshape to 8×8 and show it with `plt.imshow`. Which region of the image does the tree rely on?


In [ ]:
# 1. Fit a decision tree on the unscaled digits data.
digit_tree = DecisionTreeClassifier(criterion="entropy", random_state=1)
digit_tree.fit(Xd_train, yd_train)
digit_pred = digit_tree.predict(Xd_test)
print("Decision tree test accuracy:", accuracy_score(yd_test, digit_pred))
print("KNN baseline test accuracy:", accuracy_score(yd_test, knn.predict(sc.transform(Xd_test))))
print("No scaling is needed because decision trees split on feature thresholds rather than comparing distances.")

# 2. Compare a few maximum depths.
for depth in [3, 6, 10]:
    model = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=1)
    model.fit(Xd_train, yd_train)
    print(f"max_depth={depth}: test accuracy={accuracy_score(yd_test, model.predict(Xd_test)):.4f}")

# 3. Plot the 8x8 feature-importance map.
importance_map = digit_tree.feature_importances_.reshape(8, 8)
plt.figure(figsize=(5, 5))
plt.imshow(importance_map, cmap="viridis")
plt.colorbar(label="Feature importance")
plt.title("Decision tree pixel importance")
plt.xlabel("Pixel column")
plt.ylabel("Pixel row")
plt.show()

print("Most important pixel index:", int(np.argmax(digit_tree.feature_importances_)))


**Answer:** The decision tree is generally less accurate than KNN on the digits dataset. A single tree makes axis-aligned threshold splits and can be sensitive to small changes in training data, while KNN uses information from nearby examples. The importance map shows which pixels the tree used most, but it may focus on only a subset of pixels and ignore useful information elsewhere.


# Mini Project B: Reading a Tree on Text (NLP Track — continuing from Week 7)

Same six labeled sentences and same bag-of-words vectors as Week 7. A tree gives you something KNN cannot: **the words it chose to split on.**


In [ ]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']

vocab = sorted(set(w for s in sentences for w in s.lower().split()))

def to_vector(sentence, vocab):
    words = sentence.lower().split()
    return np.array([words.count(w) for w in vocab])

X_text = np.array([to_vector(s, vocab) for s in sentences])


### Student Practice

1. Fit a `DecisionTreeClassifier(criterion='entropy', random_state=1)` on `X_text`, `labels`.
2. Print the tree with `export_text(tree, feature_names=vocab)`. Which words does it split on?
3. Predict the topic of `'the striker scored a goal'` and of your own ambiguous sentence from Week 7.


In [ ]:
# Fit a readable decision tree to the bag-of-words vectors.
text_tree = DecisionTreeClassifier(criterion="entropy", random_state=1)
text_tree.fit(X_text, labels)

print(export_text(text_tree, feature_names=vocab))

test_sentences = [
    "the striker scored a goal",
    "the phone battery lasts all day"  # Example of an ambiguous-ish tech sentence
]
X_test_text = np.array([to_vector(sentence, vocab) for sentence in test_sentences])
predictions = text_tree.predict(X_test_text)
for sentence, prediction in zip(test_sentences, predictions):
    print(f"Sentence: {sentence!r} -> predicted topic: {prediction}")


**Answer:** The split words might change when more sentences are added, because the tree chooses words that best separate the training examples it sees. With only six sentences, the dataset is extremely small and may not represent how people actually write. Therefore, the tree can fit these examples well but may not generalize reliably; its predictions should be treated cautiously.


## Summary

| What we did | Why it matters |
|---|---|
| Entropy and information gain | The rule ID3 uses to choose every split |
| ID3 by hand on the tennis table | Exam-style problem, show your working |
| `DecisionTreeClassifier` and depth sweep | Third version of the bias-variance knob (degree, K, depth) |
| Mini A: digits | Tree vs KNN on the same data, plus which pixels matter |
| Mini B: text | A tree you can read, and why tiny data makes it fragile |

**Next week:** Naive Bayes, a probabilistic classifier. Same digits and sentences again.
